# Week 4: Sequence Models (RNN, LSTM, GRU, Bidirectionality)

## Learning Goals
- Revisit why `DAN + linear classifier` can miss sequence information especially for negation, contrastive conjunction, and sentiment shifts. 
- Build a basic `RNN` sentiment classifier in PyTorch.
- Upgrade the same pipeline to `LSTM` and `GRU`.
- Add bidirectional sequence processing to `LSTM`.
- Compare all models by metrics, speed, and failure cases.
    - metrics: accuracy (the benchmark dataset is pretty balanced)
    - inference speed
    - specific failure cases (deep dive)
    - summary/training limitations


Revisit the Week 3 `DAN` model with a linear classifier on IMDb sentiment classification.

We will load the saved imdb datasets, use the same seed as when we trained DAN model, and config files from Week 3 rather than rebuilding tokenization here.

In [35]:
from utils import (
                    load_json, 
                    load_torch_dataset, 
                    train_test_model, 
                    evaluate_model, 
                    get_device, 
                    clean_memory)

In [36]:
imdb_dataset_path = "./datasets/imdb_bpe_dataset.pt"
imdb_vocab_path = "./datasets/bpe_vocab.json"
imdb_vocab_config_path = "./models/vocab_config.json"

input_ids, labels, attention_masks = load_torch_dataset(imdb_dataset_path)
vocab_config = load_json(imdb_vocab_config_path)
bpe_vocab = load_json(imdb_vocab_path)

In [37]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, random_split, DataLoader

In [38]:
class DAN(nn.Module):
    def __init__(self, vocab_size, embedding_size, padding_idx, hidden_dim, n_classes, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size,padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.hidden = nn.Linear(embedding_size, hidden_dim)
        self.linear = nn.Linear(hidden_dim, n_classes)

    def forward(self, input_ids, mask):
        embedding = self.embedding(input_ids)
        mask = mask.unsqueeze(-1).float()
        masked_embedding = embedding * mask
        mean_pool = masked_embedding.sum(dim=1)/mask.sum(dim=1).clamp(min=1)
        mean_pool = self.dropout(mean_pool)
        hidden = self.hidden(mean_pool) 
        hidden = torch.relu(hidden)
        hidden = self.dropout(hidden)
        return self.linear(hidden)

In [39]:
pad_id = vocab_config["pad_id"]
vocab_size = vocab_config["vocab_size"]
embedding_size = 384
hidden_dim = embedding_size * 2 
dropout = 0.2
device = get_device()

In [40]:
dan_model = DAN(vocab_size, embedding_size, pad_id, hidden_dim, 1,dropout).to(device)
state = torch.load("models/dan_imdb.pt", map_location=device)
dan_model.load_state_dict(state)
dan_model = dan_model.to(device)
dan_model.eval()
for param in dan_model.parameters():
    param.requires_grad_(False)

### Sequence Models
Sequence models process text token by token, allowing the model to use word order and evolving context when making predictions. Unlike order-agnostic approaches such as DAN, sequence models can distinguish between sentences that contain similar words but different meanings because of negation, contrast, or sentiment shifts.

A basic RNN updates a hidden state over time, but it can struggle to preserve important information across long sequences. GRU and LSTM improve this by adding gating mechanisms that control what information should be kept, updated, or forgotten. LSTM goes one step further by maintaining a separate cell state for longer-term memory. Bidirectional models extend this idea by reading the sequence in both directions, so each prediction can use both left and right context.

In practice, sequence models are useful when meaning depends on composition rather than just word presence. This makes them especially helpful for examples involving negation, contrastive conjunctions, and changes in sentiment across a sentence.

* RNNs can struggle on long sequences because each hidden state depends recursively on the previous hidden state and the current input token. During backpropagation, gradients must pass through many repeated steps, which can cause them to shrink or explode. As a result, basic RNNs often have difficulty preserving important information over long distances, making them weaker on long-range dependencies and sentiment shifts later in the sentence.

* GRU improves the long-sequence limitations of a basic RNN by adding two gates that help manage memory over time. The reset gate $r_t$ controls how much of the previous hidden state should be used when computing the candidate hidden state $\bar{h}_t$. The update gate $z_t$ then decides how much of the candidate state should be used versus how much of the old hidden state should be preserved. This gives the model better control over memory and helps information flow more effectively through long sequences, which can also make optimization easier during gradient descent.A basic RNN can suffer from vanishing or exploding gradients because backpropagation through time involves repeated products of partial derivatives across many recurrent steps. In contrast, the GRU update
$h_t = (1 - z_t)\odot h_{t-1} + z_t \odot \bar{h}_t$
introduces a more direct additive path from the previous hidden state to the current hidden state. This makes the partial derivative with respect to the previous state easier to preserve, helping gradients flow backward through long sequences more effectively.

* LSTM improves long-term memory by introducing a separate cell state $c_t$, which acts as a dedicated memory path across time. The cell state is updated additively:
$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$.
Because the previous cell state can be carried forward directly through the forget gate, important information can persist over many time steps without being repeatedly overwritten. This also helps gradient flow during backpropagation, since the partial derivative through the cell state can remain large when the forget gate is close to 1. The output gate then controls how much of this memory is exposed in the hidden state $h_t$.


#### Sequence Model Comparison

| Model | Core Update | Gates / States | Intuition |
| --- | --- | --- | --- |
| RNN | $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$ | Hidden state: $h_t$ | Rewrites one memory at every step |
| GRU | $z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$  <br> $r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$  <br> $\bar{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$  <br> $h_t = (1 - z_t)\odot h_{t-1} + z_t \odot \bar{h}_t$ | Update gate: $z_t$  <br> Reset gate: $r_t$  <br> Hidden state: $h_t$  <br> Candidate hidden state: $\bar{h}_t$ | Uses gates to control how much old memory to keep and how much new information to use |
| LSTM | $f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)$  <br> $i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)$  <br> $\tilde{c}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$  <br> $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$  <br> $o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)$  <br> $h_t = o_t \odot \tanh(c_t)$ | Forget gate: $f_t$  <br> Input gate: $i_t$  <br> Output gate: $o_t$  <br> Cell state: $c_t$  <br> Hidden state: $h_t$  <br> Candidate cell state: $\tilde{c}_t$ | Adds a dedicated memory path and gates to forget, write, and expose information |

**Gate meanings**
- $z_t$: update gate in GRU, controls how much of the candidate state replaces the old state
- $r_t$: reset gate in GRU, controls how much of the previous hidden state is used to compute the candidate state
- $f_t$: forget gate in LSTM, controls how much of the old cell state is retained
- $i_t$: input gate in LSTM, controls how much new candidate cell information is written
- $o_t$: output gate in LSTM, controls how much of the cell state is exposed as the hidden state

**Notation**
- $x_t$: input at time step $t$
- $h_{t-1}$: previous hidden state
- $h_t$: current hidden state
- $c_t$: cell state in LSTM
- $\sigma$: sigmoid activation
- $\tanh$: tanh activation
- $\odot$: elementwise multiplication


In [41]:
import torch 
from torch import nn

class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_size,num_layers, dropout=0.2, num_classes=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.RNN(
            embed_dim,
            hidden_size,
            num_layers,
            batch_first=True,
            nonlinearity="relu"
        )
        self.linear = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, mask):
        lengths = mask.sum(dim=1).cpu()
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )
        _, hidden = self.rnn(packed)
        output = hidden[-1]
        output = self.dropout(output) 
        return self.linear(output) 

In [45]:
hidden_size = 256
num_layers = 1 
device = get_device()
rnn_model = RNNModel(vocab_size, embedding_size, pad_id, hidden_size, num_layers).to(device)
# rnn_model.embedding.weight.data.copy_(dan_model.embedding.weight.data)
criterion = nn.BCEWithLogitsLoss()
# patience = 5
# num_epochs = 30

In [43]:
dataset = TensorDataset(input_ids, attention_masks, labels.float())
n = len(dataset)
n_train = int(0.8*n)
n_val = int(0.1*n)
n_test = n - n_train - n_val
g = torch.Generator().manual_seed(204)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

In [10]:
clean_memory()

Memory cleaned.


In [11]:
# best_model = train_test_model(
#     device = device,
#     model = rnn_model,
#     criterion=criterion,
#     train_loader = train_loader,
#     val_loader = val_loader,
#     patience = patience,
#     num_epochs=num_epochs,
#     lr=1e-4,
#     weight_decay=1e-5,
#     is_binary=True,
#     save_path="models/rnn_imdb"
# )

In [12]:
# evaluate_model(model=best_model, data_loader=test_loader, criterion=criterion, device=device, is_binary=True)

In [14]:
# torch.save(best_model.state_dict(), "models/final_rnn_imdb.pt")
rnn_state = torch.load("models/final_rnn_imdb.pt", map_location=device)
rnn_model.load_state_dict(rnn_state)
rnn_model.eval()
for param in rnn_model.parameters():
    param.requires_grad_(False)

In [15]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_size,num_layers, dropout=0.2, num_classes=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(
            embed_dim,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.linear = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, mask):
        lengths = mask.sum(dim=1).cpu()
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        output = hidden[-1]
        output = self.dropout(output) 
        return self.linear(output) 

hidden_size = 256
num_layers = 1 
device = get_device()
gru_model = GRUModel(
   vocab_size, 
    embedding_size, 
    pad_id,
    hidden_size,
    num_layers
).to(device)
# gru_model.embedding.weight.data.copy_(dan_model.embedding.weight.data)
# criterion = nn.BCEWithLogitsLoss()
# patience = 5
# num_epochs = 30
# clean_memory()

In [16]:
# best_model = train_test_model(
#     device = device,
#     model = gru_model,
#     criterion=criterion,
#     train_loader = train_loader,
#     val_loader = val_loader,
#     patience = patience,
#     num_epochs=num_epochs,
#     lr=1e-4,
#     weight_decay=1e-5,
#     is_binary=True,
#     save_path="models/gru_imdb"
# )

In [17]:
# torch.save(best_model.state_dict(), "models/final_gru_imdb.pt")

In [18]:
# evaluate_model(model=best_model, data_loader=test_loader, criterion=criterion, device=device, is_binary=True)

In [19]:
# torch.save(best_model.state_dict(), "models/final_rnn_imdb.pt")
gru_state = torch.load("models/final_gru_imdb.pt", map_location=device)
gru_model.load_state_dict(gru_state)
gru_model.eval()
for param in gru_model.parameters():
    param.requires_grad_(False)

In [20]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_size,num_layers, dropout=0.2, num_classes=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.linear = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, mask):
        lengths = mask.sum(dim=1).cpu()
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )
        _, (hidden, cell) = self.lstm(packed)
        output = hidden[-1]
        output = self.dropout(output) 
        return self.linear(output) 

hidden_size = 256
num_layers = 1 
device = get_device()
lstm_model = LSTMModel(
   vocab_size, 
    embedding_size, 
    pad_id,
    hidden_size,
    num_layers
).to(device)
# lstm_model.embedding.weight.data.copy_(dan_model.embedding.weight.data)
# criterion = nn.BCEWithLogitsLoss()
# patience = 5
# num_epochs = 30
# clean_memory()

In [21]:
# best_model = train_test_model(
#     device = device,
#     model = lstm_model,
#     criterion=criterion,
#     train_loader = train_loader,
#     val_loader = val_loader,
#     patience = patience,
#     num_epochs=num_epochs,
#     lr=1e-4,
#     weight_decay=1e-5,
#     is_binary=True,
#     save_path="models/lstm_imdb"
# )

In [22]:
# torch.save(best_model.state_dict(), "models/final_lstm_imdb.pt")

In [23]:
# evaluate_model(model=best_model, data_loader=test_loader, criterion=criterion, device=device, is_binary=True)

In [24]:
# torch.save(best_model.state_dict(), "models/final_rnn_imdb.pt")
lstm_state = torch.load("models/final_lstm_imdb.pt", map_location=device)
lstm_model.load_state_dict(lstm_state)
lstm_model.eval()
for param in lstm_model.parameters():
    param.requires_grad_(False)

### Bidirectionality

Bidirectionality improves sequence modeling by letting the model process text both left-to-right and right-to-left, so each representation can use both past and future context. This does not directly fix vanishing gradients in the way GRU or LSTM gating does, since each direction still performs backpropagation through time. However, it can make learning easier in practice because important evidence can be captured from both directions instead of relying on a single recurrent path.

Extend the sequence model pipeline to process text both left-to-right and right-to-left with `BiLSTM`.

In a BiLSTM, the loss is backpropagated through both the forward and backward LSTM branches. Each branch performs backpropagation through time independently, and their gradients are combined through the final classifier. Bidirectionality improves access to context from both directions, while the LSTM cell state in each branch helps preserve gradients over longer sequences. BiLSTM is more computationally expensive than a unidirectional LSTM because it maintains two separate recurrent chains, one forward and one backward. During training, gradients must be computed through both branches, and their final hidden states are combined before prediction, increasing both memory usage and computation cost.


In [25]:
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_size,num_layers, dropout=0.2, num_classes=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )
        self.linear = nn.Linear(hidden_size*2, num_classes)

    def forward(self, input_ids, mask):
        lengths = mask.sum(dim=1).cpu()
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )
        _, (hidden, cell) = self.lstm(packed)
        last_backward = hidden[-1]
        last_forward = hidden[-2]
        output = torch.cat([last_forward, last_backward], dim = 1) 
        output = self.dropout(output) 
        return self.linear(output) 

hidden_size = 256
num_layers = 1 
device = get_device()
Bilstm_model = BiLSTMModel(
   vocab_size, 
    embedding_size, 
    pad_id,
    hidden_size,
    num_layers
).to(device)
# Bilstm_model.embedding.weight.data.copy_(dan_model.embedding.weight.data)
# criterion = nn.BCEWithLogitsLoss()
# patience = 5
# num_epochs = 30
# clean_memory()

In [26]:
# best_model = train_test_model(
#     device = device,
#     model = Bilstm_model,
#     criterion=criterion,
#     train_loader = train_loader,
#     val_loader = val_loader,
#     patience = patience,
#     num_epochs=num_epochs,
#     lr=1e-4,
#     weight_decay=1e-5,
#     is_binary=True,
#     save_path="models/bilstm_imdb"
# )

In [27]:
# torch.save(best_model.state_dict(), "models/final_bilstm_imdb.pt")

In [28]:
# evaluate_model(model=best_model, data_loader=test_loader, criterion=criterion, device=device, is_binary=True)

In [29]:
bilstm_state = torch.load("models/final_bilstm_imdb.pt", map_location=device)
Bilstm_model.load_state_dict(bilstm_state)
Bilstm_model.eval()
for param in Bilstm_model.parameters():
    param.requires_grad_(False)

## Day 5: Prediction Challenge Set

Create a small handwritten challenge set with:
- negation
- contrast
- sentiment shift
- scope 

This is a behavioral probe, not a formal benchmark. Do note the sequence models may not do well given the limitation of training datasets

* We run one warm-up pass before timing because the first inference can include extra one-time runtime overhead. This makes the measured inference speed more representative of steady-state model performance.

In [30]:
benchmark_set = {
    "negation": [
        {"text": "This movie is not good.", "label": 0},
        {"text": "This movie is bad.", "label": 0},
        {"text": "This movie is not bad.", "label": 1},
        {"text": "I did not enjoy this film.", "label": 0},
        {"text": "The plot was not convincing.", "label": 0},
    ],
    "contrast": [
        {"text": "The acting was weak, but the ending was fantastic.", "label": 1},
        {"text": "The acting was fantastic, but the ending was weak.", "label": 0},
        {"text": "The first half drags, but the film becomes excellent.", "label": 1},
        {"text": "The first half is excellent, but the film becomes dull.", "label": 0},
    ],
    "sentiment_shift": [
        {"text": "I expected to hate this movie, but I loved it.", "label": 1},
        {"text": "I expected to love this movie, but I hated it.", "label": 0},
        {"text": "It begins like a disaster and ends beautifully.", "label": 1},
        {"text": "It begins beautifully and ends like a disaster.", "label": 0}
    ],
    "scope": [
        {"text": "Only the final scene is good.", "label": 0},
        {"text": "The final scene is good.", "label": 1},
        {"text": "Only the soundtrack is memorable.", "label": 0},
        {"text": "The soundtrack is memorable.", "label": 1},
        {"text": "Only the ending works.", "label": 0},
        {"text": "The ending works.", "label": 1},
    ],
}


In [31]:
from utils import bpe_batch_encode
import time

bpe_merges_path = "./datasets/bpe_merges.json"
bpe_merges = load_json(bpe_merges_path)
pad_token = vocab_config["pad_token"]
max_seq = vocab_config["max_seq"]

all_sentences = [sample["text"] for test in benchmark_set.values() for sample in test]
labels_cat = [(sample["label"], key) for key, value in  benchmark_set.items() for sample in value]
labels = torch.tensor([item[0] for item in labels_cat], dtype=torch.float32)
cat = [item[1] for item in labels_cat]
input_ids, padding_masks = bpe_batch_encode(
    sentences=all_sentences,
    bpe_merges=bpe_merges,
    bpe_vocab=bpe_vocab,
    max_seq=max_seq,
    pad_token=pad_token
)
input_ids, padding_masks, labels = input_ids.to(device), padding_masks.to(device), labels.to(device)
models = [dan_model, rnn_model, gru_model, lstm_model, Bilstm_model]
# Warm-up pass
with torch.no_grad():
    for model in models:
        _ = model(input_ids, padding_masks)
    if torch.backends.mps.is_available():
        torch.mps.synchronize()
        
model_names = ["DAN", "RNN", "GRU", "LSTM", "BiLSTM"]
accuracy = []
wrong_questions = []
speed = []
parameter_counts = []
for model in models:
    parameter_counts.append(sum(p.numel() for p in model.parameters()))
    with torch.no_grad():
        start = time.perf_counter()
        logits = model(input_ids, padding_masks)
        if torch.backends.mps.is_available():
            torch.mps.synchronize()
        end = time.perf_counter()
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(-1)
        wrong_idx = (preds != labels).nonzero(as_tuple=True)[0].cpu()
        wrong_questions.append(wrong_idx)
        accuracy.append((preds ==labels).float().mean().item())
        speed.append((end-start) * 1000) # milli seconds     

In [32]:
import pandas as pd
comparison_table = pd.DataFrame(
    {"model names": model_names,
     "accuracy": accuracy,
     "speed_ms": speed,
     "parameter_counts": parameter_counts
    }
)
comparison_table

,model names,accuracy,speed_ms,parameter_counts
0,DAN,0.526316,1.058292,523009
1,RNN,0.631579,6.232584,391169
2,GRU,0.473684,8.639042,719873
3,LSTM,0.526316,7.980708,884225
4,BiLSTM,0.421053,12.762667,1541889


### Looking at the result for validation set

In [47]:
for i, model in enumerate(models):
    print(model_names[i])
    print(evaluate_model(model=model, data_loader=test_loader, criterion=criterion, device=device, is_binary=True))

DAN
(0.5750682480609455, 0.6914)
RNN
(0.5283837844959844, 0.726)
GRU
(0.49363551738734446, 0.7562)
LSTM
(0.5157197243489396, 0.741)
BiLSTM
(0.5247567316499381, 0.7432)


### Deep dive on texts that each model predicted wrong

In [27]:
for i, wrong_indices in enumerate(wrong_questions):
    print(model_names[i])
    print("---"*36)
    for ind in wrong_indices:
        idx = ind.item()
        print(cat[idx], "|", all_sentences[idx]) 
    print("---"*36) 

DAN
------------------------------------------------------------------------------------------------------------
negation | This movie is not good.
negation | This movie is not bad.
contrast | The acting was fantastic, but the ending was weak.
contrast | The first half is excellent, but the film becomes dull.
sentiment_shift | I expected to love this movie, but I hated it.
sentiment_shift | It begins beautifully and ends like a disaster.
scope | Only the final scene is good.
scope | Only the soundtrack is memorable.
scope | The ending works.
------------------------------------------------------------------------------------------------------------
RNN
------------------------------------------------------------------------------------------------------------
negation | This movie is not bad.
negation | I did not enjoy this film.
contrast | The acting was fantastic, but the ending was weak.
contrast | The first half is excellent, but the film becomes dull.
sentiment_shift | It begins b

### Summary

* Why the basic RNN performed best:
  The basic RNN had the smallest parameter count among the sequence models, which likely made it easier to optimize on the available training data. Because it is simpler, it may have adapted more effectively during training rather than overfitting to limited sequence-sensitive patterns. It also benefited from the pretrained DAN embedding, while still having enough flexibility to learn a lightweight sequential signal over the full training run.

* Why GRU, LSTM, and BiLSTM did worse:
  Although these models have stronger architectural capacity, they also introduce many more parameters and more complex optimization dynamics. On a dataset like IMDb, where many labels can already be predicted from strong sentiment-bearing words, the extra capacity may not be sufficiently rewarded. In addition, initializing from DAN embeddings may bias all recurrent models toward lexical sentiment cues learned by the baseline. Larger gated models then face the harder problem of not only learning sequence behavior, but also overcoming a starting point that already works reasonably well without deep compositional reasoning.

* Why BiLSTM did not clearly win:
  Bidirectionality provides access to both left and right context, but it also increases computation, parameter count, and optimization difficulty. On a small handcrafted benchmark and a training dataset that does not strongly emphasize order-sensitive reasoning, the added complexity may not translate into better results. The model may have more representational power, but that power is only useful if the training data consistently rewards using it.

* Why none of the sequence models performed especially well on order-sensitive examples:
  The main issue is likely the mismatch between training signal and evaluation objective. IMDb sentiment classification often rewards lexical cues such as “great,” “terrible,” or “boring,” so order-agnostic models can already perform strongly. In contrast, the benchmark focuses on negation, contrast, reversal, and scope, which require compositional reasoning and more careful use of context. These patterns are relatively sparse in the training signal, so having an architecture capable of modeling sequence does not automatically mean the model will learn that behavior reliably.

* Main takeaway:
  This comparison suggests that model architecture alone is not enough. Sequence models provide the capacity to model order, but they still depend on training data that makes order matter often enough for the model to learn it. When the dataset is dominated by strong lexical sentiment cues, simpler baselines such as DAN can remain highly competitive, and more expressive recurrent models may not show their full advantage.

